# Global Linear RRF Weights With LLM Judge

This notebook implements the exact global-weight experiment:

1. Regenerate the four **base** agent query variants with Ollama (`output_with_agents_<uc>.csv`).
2. Split UC1 and UC2 into training and validation queries.
3. Run BM25 + cross-encoder candidate ranking and RRF.
4. Let the Ollama judge improve the four query variants (`ollama_global_judge_*` columns; one DeepSeek call per variant).
5. Run BM25 + cross-encoder candidate ranking again and RRF.
6. Use the resulting rankings to learn one global set of RRF weights.
7. Apply the full judged pipeline to test data: the UC1/UC2 validation halves and all UC3 queries.
8. Evaluate recall, precision, and accuracy.

Only one weight vector is learned across UC1 and UC2, so the model is more generalizable than per-use-case or per-level weights.

## Method

For each original query $q$, the pipeline uses five retrieval inputs:

$$
V = \{v_0, v_1, v_2, v_3, v_4\}
$$

where $v_0$ is the original query and $v_1,\dots,v_4$ are the legal terminology, compliance, contract, and risk variants.

For each variant $v_i$, BM25 retrieves candidates and the cross-encoder reranks those candidates with the original query $q$. The rank-based feature is:

$$
x_i(q,d) =
\begin{cases}
\frac{1}{K + r_i(q,d)} & \text{if document } d \text{ is retrieved by variant } v_i \\
0 & \text{otherwise}
\end{cases}
$$

A single global logistic regression is trained over all UC1/UC2 training examples:

$$
P(y=1 \mid q,d) = \sigma\left(\beta_0 + \boldsymbol{\beta}^\top \mathbf{x}(q,d)\right)
$$

The final RRF weights are obtained by clipping negative coefficients and normalizing:

$$
w_i = \frac{\max(\beta_i, 0)}{\sum_j \max(\beta_j, 0)}
$$

Final ranking uses:

$$
s(q,d) = \sum_i w_i x_i(q,d)
$$

In [1]:
from pathlib import Path
import importlib
import os
import sys

import numpy as np
import pandas as pd
import requests
from sklearn.linear_model import LogisticRegression

project_root = Path.cwd()
while project_root.name != "Legal-Query-Synthesis-from-Business-Processes-for-Agentic-Retrieval" and project_root.parent != project_root:
    project_root = project_root.parent

os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import main
importlib.reload(main)

from llm_provider import OllamaProvider
from main import JudgeContext, QueryVariants, clean_text
from retrieval.retrieval_bm25 import Query, build_bm25_index, load_corpus
from retrieval.sota_retrieval import SotaRetriever

print(f"Working directory: {Path.cwd()}")

Working directory: /Users/mareklorenz/Development/Legal-Query-Synthesis-from-Business-Processes-for-Agentic-Retrieval


## Configuration

**Base agent variants** (`legal_terminology_rewrite`, …) are always regenerated with Ollama (`main.ensure_query_variants`, `force_regenerate=True`) into `output_with_agents_<uc>.csv` before the judge step.

`RUN_LLM_JUDGE` controls whether judged variants are generated with Ollama. The notebook stores Ollama outputs in provider-specific `ollama_global_judge_*` columns, so older Gemini `global_judge_*` columns are not reused when recalculating weights.

In [2]:
TRAIN_USE_CASES = ["uc1", "uc2"]
EXTERNAL_TEST_USE_CASES = ["uc3"]
ALL_USE_CASES = TRAIN_USE_CASES + EXTERNAL_TEST_USE_CASES
LEVELS = ["process", "subprocess", "task"]

RRF_K = 60
RANDOM_STATE = 42
VARIANT_PROVIDER = "ollama"
JUDGE_PROVIDER = "ollama"
OLLAMA_MODEL = os.environ.get("OLLAMA_MODEL", "deepseek-r1:latest")
OLLAMA_BASE_URL = os.environ.get("OLLAMA_BASE_URL", "http://localhost:11434")
RUN_LLM_JUDGE = True
FORCE_REGENERATE_JUDGE_VARIANTS = True
SAVE_JUDGE_PROGRESS_EVERY = 1

BASE_VARIANTS = [
    ("baseline", "query"),
    ("legal_terminology_rewrite", "legal_terminology_rewrite"),
    ("regulatory_compliance_query", "regulatory_compliance_query"),
    ("contract_clause_query", "contract_clause_query"),
    ("risk_scenario_query", "risk_scenario_query"),
]

JUDGE_COLUMN_PREFIX = f"{JUDGE_PROVIDER}_global_judge"
JUDGED_VARIANT_FIELDS = [
    "legal_terminology_rewrite",
    "regulatory_compliance_query",
    "contract_clause_query",
    "risk_scenario_query",
]

GLOBAL_JUDGE_VARIANTS = [("baseline", "query")] + [
    (f"{JUDGE_COLUMN_PREFIX}_{field}", f"{JUDGE_COLUMN_PREFIX}_{field}")
    for field in JUDGED_VARIANT_FIELDS
]

LEVEL_TO_GS_SUFFIX = {
    "process": "process_level",
    "subprocess": "subprocess_level",
    "task": "event_level",
}

LEVEL_TO_TOP_K = {
    "process": 100,
    "subprocess": 30,
    "task": 15,
}

INITIAL_RRF_WEIGHTS = np.ones(len(BASE_VARIANTS)) / len(BASE_VARIANTS)

In [3]:
def corpus_path_for(use_case):
    return Path(f"regulatory_relevance4process-D73C/SOTA_NLP_LIR/input_ranking/{use_case}/Input_corpus_{use_case}.xlsx")


def gold_path_for(use_case, level):
    suffix = LEVEL_TO_GS_SUFFIX[level]
    return Path(f"regulatory_relevance4process-D73C/SOTA_NLP_LIR/output_ranking_input_eval/{use_case}/gold_standard/gs_{use_case}_{suffix}.xlsx")


def recorded_path_for(use_case):
    return Path(f"output_with_agents_{use_case}.csv")


def assert_ollama_ready():
    if JUDGE_PROVIDER != "ollama":
        raise ValueError("This notebook is configured to run the judge with Ollama only.")

    tags_url = f"{OLLAMA_BASE_URL.rstrip('/')}/api/tags"
    try:
        response = requests.get(tags_url, timeout=10)
        response.raise_for_status()
    except requests.RequestException as exc:
        raise RuntimeError(
            f"Ollama is not reachable at {OLLAMA_BASE_URL}. Start Ollama before running this notebook."
        ) from exc

    available_models = {model.get("name") for model in response.json().get("models", [])}
    if OLLAMA_MODEL not in available_models:
        raise RuntimeError(
            f"Ollama model '{OLLAMA_MODEL}' is not available. Run: ollama pull {OLLAMA_MODEL}"
        )


def bind_ollama_for_agents():
    """Route agent variant LLM calls through this notebook's Ollama model."""
    from llm_provider import OllamaProvider
    from llm_provider import get_llm_provider as _orig_get_llm_provider

    def _get_llm_provider(provider: str = "ollama", **kwargs):
        if provider.lower() == "ollama":
            return OllamaProvider(model=OLLAMA_MODEL, base_url=OLLAMA_BASE_URL)
        return _orig_get_llm_provider(provider, **kwargs)

    import llm_provider
    import agents.clause_contract_agent as contract_mod
    import agents.legal_terminology_rewriter as legal_mod
    import agents.regulatory_compliance_agent as reg_mod
    import agents.scenario_risk_agent as risk_mod

    llm_provider.get_llm_provider = _get_llm_provider
    for mod in (legal_mod, reg_mod, contract_mod, risk_mod):
        mod.get_llm_provider = _get_llm_provider


def ensure_base_query_variants(use_case: str) -> Path:
    """Always regenerate the four agent query variants with Ollama."""
    bind_ollama_for_agents()
    assert_ollama_ready()
    print(f"{use_case}: regenerating base agent variants with {OLLAMA_MODEL}.")
    return main.ensure_query_variants(
        use_case=use_case,
        provider=VARIANT_PROVIDER,
        fill_missing_only=False,
        force_regenerate=True,
    )


def load_records(use_case):
    path = recorded_path_for(use_case)
    if not path.exists():
        raise FileNotFoundError(
            f"Missing recorded query variants: {path}. "
            "Run the 'Generate base query variants' cell first."
        )
    df = pd.read_csv(path)
    df["level"] = df["level"].astype(str).str.lower()
    df["query_clean"] = df["query"].apply(clean_text)
    return df


def save_records(use_case, records_df):
    path = recorded_path_for(use_case)
    records_df.drop(columns=["query_clean"], errors="ignore").to_csv(path, index=False)
    print(f"Saved {path}")


def variant_text(row, column):
    if column not in row.index:
        return clean_text(row["query"])
    value = row[column]
    if value is None or (isinstance(value, float) and pd.isna(value)) or str(value).strip() == "":
        return clean_text(row["query"])
    return clean_text(value)


def split_queries(queries):
    queries = list(queries)
    if len(queries) == 1:
        return queries, queries
    rng = np.random.default_rng(RANDOM_STATE)
    indices = np.arange(len(queries))
    rng.shuffle(indices)
    split_at = max(1, len(indices) // 2)
    train_indices = set(indices[:split_at])
    train_queries = [query for idx, query in enumerate(queries) if idx in train_indices]
    validation_queries = [query for idx, query in enumerate(queries) if idx not in train_indices]
    return train_queries, validation_queries

## BM25 + CE + RRF Helpers

Each variant retrieves BM25 candidates. The cross-encoder reranks those candidates with the original query. RRF then combines the variant lists.

In [4]:
def ce_rank_variant_lists(row, variant_columns, bm25_index, documents, cross_encoder, top_k):
    original_query = clean_text(row["query"])
    ranked_lists = {}
    for variant_name, column in variant_columns:
        bm25_results = bm25_index.rank(Query(text=variant_text(row, column)), top_k=top_k)
        if not bm25_results:
            ranked_lists[variant_name] = []
            continue
        cross_inputs = [[original_query, documents[result.document.doc_id].text] for result in bm25_results]
        cross_scores = cross_encoder.predict(cross_inputs)
        scored = [
            (result.document.doc_id, float(score))
            for result, score in zip(bm25_results, cross_scores, strict=True)
        ]
        scored.sort(key=lambda item: item[1], reverse=True)
        ranked_lists[variant_name] = [
            {"doc_id": doc_id, "score": score, "rank": rank}
            for rank, (doc_id, score) in enumerate(scored, start=1)
        ]
    return ranked_lists


def rrf_feature_rows(ranked_lists, variant_columns):
    doc_features = {}
    for variant_index, (variant_name, _) in enumerate(variant_columns):
        for item in ranked_lists.get(variant_name, []):
            features = doc_features.setdefault(item["doc_id"], np.zeros(len(variant_columns), dtype=float))
            features[variant_index] = 1.0 / (RRF_K + item["rank"])
    return doc_features


def rank_with_rrf(row, variant_columns, weights, bm25_index, documents, cross_encoder, top_k):
    ranked_lists = ce_rank_variant_lists(row, variant_columns, bm25_index, documents, cross_encoder, top_k)
    doc_features = rrf_feature_rows(ranked_lists, variant_columns)
    scored_docs = [(doc_id, float(np.dot(features, weights))) for doc_id, features in doc_features.items()]
    scored_docs.sort(key=lambda item: item[1], reverse=True)
    return scored_docs[:top_k], ranked_lists, doc_features


def ranked_docs_to_context_df(scored_docs, documents, query, level):
    return pd.DataFrame(
        [
            {
                "level": level,
                "query": query,
                "rel_text": clean_text(documents[doc_id].text),
                "score": score,
                "method": "rrf_ce_context",
                "query_variant": "rrf_ce_context",
                "source_variants": "rrf_ce_context",
            }
            for doc_id, score in scored_docs
        ]
    )

## Ollama judge query improvement (DeepSeek, one call per variant)

DeepSeek gets **four separate prompts** (legal / regulatory / contract / risk). Each must return **only one rewritten query line** for BM25 recall. Implemented in `agents.query_judge_deepseek`. With `FORCE_REGENERATE_JUDGE_VARIANTS = True`, every row is re-judged before weights are learned.

## Generate base query variants (Ollama)

Regenerates all four agent rewrites for every query in UC1–UC3 (`force_regenerate=True`) and writes them to `output_with_agents_<uc>.csv`. Run this before the training cell (which runs the Ollama judge on those variants).

In [5]:
def base_variants_from_row(row):
    return QueryVariants(
        legal_terminology_rewrite=variant_text(row, "legal_terminology_rewrite"),
        regulatory_compliance_query=variant_text(row, "regulatory_compliance_query"),
        contract_clause_query=variant_text(row, "contract_clause_query"),
        risk_scenario_query=variant_text(row, "risk_scenario_query"),
    )


def judged_variant_columns():
    return [column for _, column in GLOBAL_JUDGE_VARIANTS if column != "query"]


def judge_columns_missing(row):
    return any(
        column not in row.index or pd.isna(row[column]) or str(row[column]).strip() == ""
        for column in judged_variant_columns()
    )


from agents.query_judge_deepseek import deepseek_judge_refine_queries as _deepseek_judge_refine_queries


def ollama_judge_refine_queries(context, documents):
    return _deepseek_judge_refine_queries(
        context,
        documents,
        model=OLLAMA_MODEL,
        base_url=OLLAMA_BASE_URL,
        recorded_value=main.recorded_value,
        query_variants_cls=QueryVariants,
    )


def improve_row_queries(row, level, bm25_index, documents, sota_retriever, top_k):
    if not RUN_LLM_JUDGE:
        raise ValueError("Missing Ollama judge variants. Set RUN_LLM_JUDGE = True to generate them.")

    initial_ranked, _, _ = rank_with_rrf(
        row=row,
        variant_columns=BASE_VARIANTS,
        weights=INITIAL_RRF_WEIGHTS,
        bm25_index=bm25_index,
        documents=documents,
        cross_encoder=sota_retriever.cross_encoder,
        top_k=top_k,
    )
    baseline_bm25 = bm25_index.rank(Query(text=clean_text(row["query"])), top_k=top_k)
    ce_context = ranked_docs_to_context_df(initial_ranked[:15], documents, clean_text(row["query"]), level)
    return ollama_judge_refine_queries(
        JudgeContext(
            original_query=clean_text(row["query"]),
            variants=base_variants_from_row(row),
            bm25_results=baseline_bm25[:15],
            ce_rows=ce_context,
        ),
        documents=documents,
    )


def write_judged_variants(records_df, row_index, judged):
    values_by_field = {
        "legal_terminology_rewrite": judged.legal_terminology_rewrite,
        "regulatory_compliance_query": judged.regulatory_compliance_query,
        "contract_clause_query": judged.contract_clause_query,
        "risk_scenario_query": judged.risk_scenario_query,
    }
    for field, value in values_by_field.items():
        records_df.loc[row_index, f"{JUDGE_COLUMN_PREFIX}_{field}"] = value


def ensure_global_judge_variants(use_case, records_df):
    for column in judged_variant_columns():
        if column not in records_df.columns:
            records_df[column] = ""

    missing_mask = records_df.apply(judge_columns_missing, axis=1)
    if FORCE_REGENERATE_JUDGE_VARIANTS:
        rows_to_judge = records_df.index
        print(f"{use_case}: regenerating {len(rows_to_judge)} Ollama judge rows with {OLLAMA_MODEL}.")
    else:
        rows_to_judge = records_df[missing_mask].index
        if len(rows_to_judge) == 0:
            print(f"{use_case}: Ollama judge variants already available.")
            return records_df
        print(f"{use_case}: generating {len(rows_to_judge)} missing Ollama judge rows with {OLLAMA_MODEL}.")

    assert_ollama_ready()
    documents = load_corpus(str(corpus_path_for(use_case)))
    bm25_index = build_bm25_index(str(corpus_path_for(use_case)))
    sota_retriever = SotaRetriever([document.text for document in documents])

    for count, (row_index, row) in enumerate(records_df.loc[rows_to_judge].iterrows(), start=1):
        level = row["level"]
        top_k = LEVEL_TO_TOP_K[level]
        print(f"[{use_case} / {level}] Ollama judge query improvement for row {row_index}")
        judged = improve_row_queries(row, level, bm25_index, documents, sota_retriever, top_k)
        write_judged_variants(records_df, row_index, judged)
        if SAVE_JUDGE_PROGRESS_EVERY and count % SAVE_JUDGE_PROGRESS_EVERY == 0:
            save_records(use_case, records_df)

    save_records(use_case, records_df)
    return records_df

In [6]:
for use_case in ALL_USE_CASES:
    ensure_base_query_variants(use_case)

uc1: regenerating base agent variants with deepseek-r1:latest.
Loaded 1 queries for level 'process'
Loaded 7 queries for level 'subprocess'
Loaded 31 queries for level 'task'
[uc1 / process] Generating variants for: The process for a travel insurance claim involves several important steps to ensure a fair...
  Generating missing variant: legal_terminology_rewrite
  Generating missing variant: regulatory_compliance_query
  Generating missing variant: contract_clause_query
  Generating missing variant: risk_scenario_query
[uc1 / subprocess] Generating variants for: There are various options for submitting a claim and logging the information into the comp...
  Generating missing variant: legal_terminology_rewrite
  Generating missing variant: regulatory_compliance_query
  Generating missing variant: contract_clause_query
  Generating missing variant: risk_scenario_query
[uc1 / subprocess] Generating variants for: The claims consultant is reviewing the information received for a travel ins

## Split UC1/UC2 And Learn One Global Weight Vector

The training examples come from all UC1/UC2 training queries across all levels. This produces one shared vector $\mathbf{w}$.

In [ ]:
def build_gold_lookup(use_case, level, documents):
    gold_df = pd.read_excel(gold_path_for(use_case, level))
    gold_df["query_clean"] = gold_df["query"].apply(clean_text)
    doc_id_by_text = {clean_text(document.text): document.doc_id for document in documents}

    gold_lookup = {}
    for query, group in gold_df.groupby("query_clean"):
        gold_doc_ids = {
            doc_id_by_text[clean_text(rel_text)]
            for rel_text in group["rel_text"].tolist()
            if clean_text(rel_text) in doc_id_by_text
        }
        gold_lookup[query] = gold_doc_ids
    return gold_lookup


def level_queries(records_df, gold_lookup, level):
    level_records = records_df[records_df["level"] == level]
    return [query for query in level_records["query_clean"].tolist() if query in gold_lookup]


def build_examples_for_queries(use_case, level, records_df, query_subset):
    top_k = LEVEL_TO_TOP_K[level]
    documents = load_corpus(str(corpus_path_for(use_case)))
    bm25_index = build_bm25_index(str(corpus_path_for(use_case)))
    sota_retriever = SotaRetriever([document.text for document in documents])
    gold_lookup = build_gold_lookup(use_case, level, documents)
    level_records = records_df[records_df["level"] == level].copy()

    examples = []
    labels = []
    for query in query_subset:
        row = level_records[level_records["query_clean"] == query].iloc[0]
        _, _, doc_features = rank_with_rrf(
            row=row,
            variant_columns=GLOBAL_JUDGE_VARIANTS,
            weights=INITIAL_RRF_WEIGHTS,
            bm25_index=bm25_index,
            documents=documents,
            cross_encoder=sota_retriever.cross_encoder,
            top_k=top_k,
        )
        positives = gold_lookup.get(query, set())
        for doc_id, features in doc_features.items():
            examples.append(features)
            labels.append(1 if doc_id in positives else 0)
    return examples, labels


records_by_use_case = {}
splits = []
all_examples = []
all_labels = []

for use_case in TRAIN_USE_CASES:
    records_df = ensure_global_judge_variants(use_case, load_records(use_case))
    records_by_use_case[use_case] = records_df
    documents = load_corpus(str(corpus_path_for(use_case)))
    for level in LEVELS:
        gold_lookup = build_gold_lookup(use_case, level, documents)
        queries = level_queries(records_df, gold_lookup, level)
        train_queries, validation_queries = split_queries(queries)
        splits.append(
            {
                "use_case": use_case,
                "level": level,
                "train_queries": train_queries,
                "validation_queries": validation_queries,
            }
        )
        examples, labels = build_examples_for_queries(use_case, level, records_df, train_queries)
        all_examples.extend(examples)
        all_labels.extend(labels)

x_train = np.array(all_examples)
y_train = np.array(all_labels)
print(f"Training examples: {len(x_train)}")
print(f"Positive labels: {int(y_train.sum())}")

uc1: regenerating 39 Ollama judge rows with deepseek-r1:latest.
[uc1 / process] Ollama judge query improvement for row 0
[DeepSeek judge / legal_terminology_rewrite] raw model output:
<think>
Okay, so I need to rewrite the original business process query into a legal terminology version for a regulatory text corpus. The original query is about handling travel insurance claims, and there are some specific steps involved.

First, I'll break down the original text. It starts by mentioning that when a traveler reports a claim, the insurer asks for detailed information, including necessary documentation. Then it talks about determining coverage under the policy, assessing the claim's validity and payout amount, and finalizing payment or denial.

I should use legal terms throughout this process. For instance, "travel insurance claim" is fine as it's common in legal contexts. Instead of saying "provider," I'll use "insurer." The steps can be rephrased with terms like "evaluating policy covera

In [ ]:
def learn_global_weights(x_train, y_train):
    if len(x_train) == 0 or len(np.unique(y_train)) < 2:
        return np.ones(len(GLOBAL_JUDGE_VARIANTS)) / len(GLOBAL_JUDGE_VARIANTS), "equal_weights_fallback"

    model = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE)
    model.fit(x_train, y_train)
    raw_weights = np.maximum(model.coef_[0], 0.0)
    if raw_weights.sum() == 0:
        return np.ones(len(GLOBAL_JUDGE_VARIANTS)) / len(GLOBAL_JUDGE_VARIANTS), "equal_weights_fallback"
    return raw_weights / raw_weights.sum(), "global_linear_logistic_coefficients"


global_weights, weight_source = learn_global_weights(x_train, y_train)
weights_df = pd.DataFrame(
    [
        {
            "weight_source": weight_source,
            "judge_provider": JUDGE_PROVIDER,
            "ollama_model": OLLAMA_MODEL,
            "judge_column_prefix": JUDGE_COLUMN_PREFIX,
            "training_use_cases": ",".join(TRAIN_USE_CASES),
            "training_examples": len(x_train),
            "positive_labels": int(y_train.sum()),
            **{variant_name: weight for (variant_name, _), weight in zip(GLOBAL_JUDGE_VARIANTS, global_weights)},
        }
    ]
)
weights_df

NameError: name 'x_train' is not defined

## Apply Pipeline To Test Data

Test data consists of:

- UC1 validation split
- UC2 validation split
- all UC3 queries

For each test query, the notebook uses the judged variants, reruns BM25 + CE for each variant, merges with global weighted RRF, and evaluates the final ranking.

In [ ]:
def evaluate_prediction_sets(predictions, gold_lookup, corpus_size):
    tp = fp = fn = tn = 0
    for query, predicted in predictions.items():
        gold = gold_lookup.get(query, set())
        tp += len(predicted & gold)
        fp += len(predicted - gold)
        fn += len(gold - predicted)
        tn += corpus_size - len(gold | predicted)

    accuracy = (tp + tn) / (tp + fp + fn + tn) if tp + fp + fn + tn else np.nan
    precision = tp / (tp + fp) if tp + fp else np.nan
    recall = tp / (tp + fn) if tp + fn else np.nan
    return {
        "true_positives": tp,
        "false_positives": fp,
        "false_negatives": fn,
        "true_negatives": tn,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
    }


def apply_global_pipeline(use_case, level, records_df, query_subset, split_name):
    top_k = LEVEL_TO_TOP_K[level]
    documents = load_corpus(str(corpus_path_for(use_case)))
    bm25_index = build_bm25_index(str(corpus_path_for(use_case)))
    sota_retriever = SotaRetriever([document.text for document in documents])
    gold_lookup = build_gold_lookup(use_case, level, documents)
    level_records = records_df[records_df["level"] == level].copy()

    predictions = {}
    rows = []
    for query in query_subset:
        row = level_records[level_records["query_clean"] == query].iloc[0]
        ranked_docs, _, _ = rank_with_rrf(
            row=row,
            variant_columns=GLOBAL_JUDGE_VARIANTS,
            weights=global_weights,
            bm25_index=bm25_index,
            documents=documents,
            cross_encoder=sota_retriever.cross_encoder,
            top_k=top_k,
        )
        predictions[query] = {doc_id for doc_id, _ in ranked_docs}
        for rank, (doc_id, score) in enumerate(ranked_docs, start=1):
            rows.append(
                {
                    "use_case": use_case,
                    "level": level,
                    "split": split_name,
                    "query": query,
                    "rank": rank,
                    "doc_id": doc_id,
                    "score": score,
                    "rel_text": clean_text(documents[doc_id].text),
                }
            )

    metrics = evaluate_prediction_sets(predictions, gold_lookup, len(documents))
    metrics.update(
        {
            "use_case": use_case,
            "level": level,
            "split": split_name,
            "queries": len(query_subset),
        }
    )
    return metrics, rows


metric_rows = []
prediction_rows = []

for split in splits:
    metrics, rows = apply_global_pipeline(
        use_case=split["use_case"],
        level=split["level"],
        records_df=records_by_use_case[split["use_case"]],
        query_subset=split["validation_queries"],
        split_name="validation",
    )
    metric_rows.append(metrics)
    prediction_rows.extend(rows)

for use_case in EXTERNAL_TEST_USE_CASES:
    records_df = ensure_global_judge_variants(use_case, load_records(use_case))
    records_by_use_case[use_case] = records_df
    documents = load_corpus(str(corpus_path_for(use_case)))
    for level in LEVELS:
        gold_lookup = build_gold_lookup(use_case, level, documents)
        queries = level_queries(records_df, gold_lookup, level)
        metrics, rows = apply_global_pipeline(
            use_case=use_case,
            level=level,
            records_df=records_df,
            query_subset=queries,
            split_name="external_test",
        )
        metric_rows.append(metrics)
        prediction_rows.extend(rows)

metrics_df = pd.DataFrame(metric_rows)
predictions_df = pd.DataFrame(prediction_rows)
metrics_df

uc3: regenerating 26 Ollama judge rows with deepseek-r1:latest.
[uc3 / process] Ollama judge query improvement for row 0
[WARN] Failed to parse Ollama judge response: Expecting value: line 1 column 1 (char 0). Keeping original query variants.
Saved output_with_agents_uc3.csv
[uc3 / subprocess] Ollama judge query improvement for row 1
[WARN] Failed to parse Ollama judge response: Expecting value: line 1 column 1 (char 0). Keeping original query variants.
Saved output_with_agents_uc3.csv
[uc3 / subprocess] Ollama judge query improvement for row 2
[WARN] Failed to parse Ollama judge response: Expecting value: line 1 column 1 (char 0). Keeping original query variants.
Saved output_with_agents_uc3.csv
[uc3 / subprocess] Ollama judge query improvement for row 3
[WARN] Failed to parse Ollama judge response: Expecting value: line 1 column 1 (char 0). Keeping original query variants.
Saved output_with_agents_uc3.csv
[uc3 / subprocess] Ollama judge query improvement for row 4
[WARN] Failed to pa

,true_positives,false_positives,false_negatives,true_negatives,accuracy,precision,recall,use_case,level,split,queries
0,18,82,31,358,0.768916,0.180000,0.367347,uc1,process,validation,1
1,8,112,30,1806,0.927403,0.066667,0.210526,uc1,subprocess,validation,4
2,16,209,57,7053,0.963736,0.071111,0.219178,uc1,task,validation,15
3,25,75,6,205,0.739550,0.250000,0.806452,uc2,process,validation,1
4,18,102,22,1102,0.900322,0.150000,0.450000,uc2,subprocess,validation,4
5,20,130,45,2915,0.943730,0.133333,0.307692,uc2,task,validation,10
6,17,83,0,68,0.505952,0.170000,1.000000,uc3,process,external_test,1
7,28,122,19,671,0.832143,0.186667,0.595745,uc3,subprocess,external_test,5
8,39,261,44,3016,0.909226,0.130000,0.469880,uc3,task,external_test,20


## Summary Table

This table reports recall, precision, and accuracy for each use case and level.

## Query diversification exports (`global_weights_judged_deepseek`)

Writes per-use-case files under `query_diversification_results/global_weights_judged_deepseek/`, matching `global_weights_judged`:

- `output_with_agents_<uc>_global_judge_ranking.csv` — judged query rewrites (four variant columns)
- `output_ranking_<uc>_global_judge.xlsx` — ranked predictions with learned global weights

Same file names as `global_weights_judged`; base Ollama rewrites remain in project-root `output_with_agents_<uc>.csv`.

Run after the training and evaluation cells (`records_by_use_case`, `predictions_df`), or from the repo root:

`python regulatory_relevance4process-D73C/SOTA_NLP_LIR/export_global_weights_judged_deepseek.py`

In [ ]:
_sota_dir = project_root / "regulatory_relevance4process-D73C/SOTA_NLP_LIR"
if str(_sota_dir) not in sys.path:
    sys.path.insert(0, str(_sota_dir))

from export_global_weights_judged_deepseek import export_global_weights_judged_deepseek

export_global_weights_judged_deepseek(records_by_use_case, predictions_df, project_root)

In [ ]:
summary_df = metrics_df[
    [
        "use_case",
        "level",
        "split",
        "queries",
        "true_positives",
        "false_positives",
        "false_negatives",
        "true_negatives",
        "recall",
        "precision",
        "accuracy",
    ]
].copy()
summary_df[["recall", "precision", "accuracy"]] = summary_df[["recall", "precision", "accuracy"]].round(3)
summary_df

,use_case,level,split,queries,true_positives,false_positives,false_negatives,true_negatives,recall,precision,accuracy
0,uc1,process,validation,1,18,82,31,358,0.367,0.180,0.769
1,uc1,subprocess,validation,4,8,112,30,1806,0.211,0.067,0.927
2,uc1,task,validation,15,16,209,57,7053,0.219,0.071,0.964
3,uc2,process,validation,1,25,75,6,205,0.806,0.250,0.740
4,uc2,subprocess,validation,4,18,102,22,1102,0.450,0.150,0.900
5,uc2,task,validation,10,20,130,45,2915,0.308,0.133,0.944
6,uc3,process,external_test,1,17,83,0,68,1.000,0.170,0.506
7,uc3,subprocess,external_test,5,28,122,19,671,0.596,0.187,0.832
8,uc3,task,external_test,20,39,261,44,3016,0.470,0.130,0.909


## Export Results

The workbook stores the single global weight vector, split definitions, evaluation metrics, and ranked predictions.

In [ ]:
export_path = project_root / f"linear_rrf_global_weights_{JUDGE_PROVIDER}_judge_results.xlsx"

split_rows = []
for split in splits:
    split_rows.append(
        {
            "use_case": split["use_case"],
            "level": split["level"],
            "train_queries": len(split["train_queries"]),
            "validation_queries": len(split["validation_queries"]),
        }
    )
splits_df = pd.DataFrame(split_rows)

with pd.ExcelWriter(export_path) as writer:
    weights_df.to_excel(writer, sheet_name="global_weights", index=False)
    splits_df.to_excel(writer, sheet_name="splits", index=False)
    metrics_df.to_excel(writer, sheet_name="metrics", index=False)
    predictions_df.to_excel(writer, sheet_name="ranked_predictions", index=False)

print(f"Exported {export_path}")

_sota_dir = project_root / "regulatory_relevance4process-D73C/SOTA_NLP_LIR"
if str(_sota_dir) not in sys.path:
    sys.path.insert(0, str(_sota_dir))

from export_global_weights_judged_deepseek import export_global_weights_judged_deepseek

export_global_weights_judged_deepseek(records_by_use_case, predictions_df, project_root)

Exported /Users/mareklorenz/Development/Legal-Query-Synthesis-from-Business-Processes-for-Agentic-Retrieval/linear_rrf_global_weights_ollama_judge_results.xlsx
